# 01 Build Company-Level Latest Financial Features

本 notebook 将 candidate accounts 的 filing/account-row 宽表整理成 **company-level latest 特征表**。

输入来自上一阶段：

- `financial_features_candidate_year.csv`
- `candidate_monthly_coverage_company_flags.csv`
- `candidate_monthly_coverage_report.json`

核心输出：

- `financial_features_candidate_latest_company.csv`
- `financial_features_candidate_latest_model_ready_t1.csv`
- `financial_features_candidate_latest_proxy_unlabelled.csv`
- `financial_features_candidate_latest_company_summary.csv`
- `financial_features_candidate_latest_missingness.csv`
- `model_feature_columns.json`

口径说明：

- 每家公司只保留 `period_end` 最新的一份 accounts filing 作为最新特征。
- `latest_has_observed_turnover` 表示最新 filing 是否有 turnover label。
- `company_has_any_observed_turnover` 表示该公司在所有 matched filing 中是否曾出现 turnover。
- 后续建模 label 默认使用 latest filing 的 observed turnover，避免旧 label 配新特征。


本版新增 `derived ratio / structure features`，用于表达资产、债权、负债、员工之间的结构关系，而不是只依赖总体 proxy score。


In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd


def find_repo_root():
    env_root = os.environ.get("PROJECT_ROOT")
    starts = []
    if env_root:
        starts.append(Path(env_root))
    starts.append(Path.cwd())

    for start in starts:
        start = start.resolve()
        for p in [start, *start.parents]:
            if (p / ".git").exists() and (p / "00 Data Preparation + EDA").exists():
                return p
    raise FileNotFoundError(
        "Cannot find repository root. Start Jupyter from inside the cloned repo, "
        "or set PROJECT_ROOT to the repository root."
    )


def require_exists(path, label="path"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def first_existing(paths, label):
    paths = [Path(path) for path in paths]
    for path in paths:
        if path.exists():
            return path
    tried = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Cannot find {label}. Tried:\n{tried}")


PROJECT_ROOT = find_repo_root()
FINANCIAL_DIR = PROJECT_ROOT / "00 Data Preparation + EDA" / "Financial Data"
SMALL_TEST_DIR = FINANCIAL_DIR / "Small Scale Data Test"
SAMPLE_COMPANIES_DIR = SMALL_TEST_DIR / "Sample Companies"
LOCAL_DATA_ROOT = Path(os.environ.get(
    "LOCAL_DATA_ROOT",
    r"E:\000硕士毕设\财务数据\Local Large Data",
))
ACCOUNTS_ZIP_DIR = first_existing(
    [
        LOCAL_DATA_ROOT / "Accounts Data_2025.1_2026.5",
        FINANCIAL_DIR / "Accounts Data_2025.1_2026.5",
    ],
    "accounts ZIP directory",
)
BULK_MATCH_DIR = SMALL_TEST_DIR / "2025全年bulk匹配"
MODEL_PREP_DIR = BULK_MATCH_DIR / "模型训练与数据处理"
MODEL_TRAIN_DIR = MODEL_PREP_DIR / "模型训练"

require_exists(FINANCIAL_DIR, "financial data directory")
require_exists(SMALL_TEST_DIR, "small scale test directory")
require_exists(ACCOUNTS_ZIP_DIR, "accounts ZIP directory")


BASE_DIR = BULK_MATCH_DIR
OUT_DIR = MODEL_PREP_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_FILE = BASE_DIR / "financial_features_candidate_year.csv"
COVERAGE_FLAGS_FILE = BASE_DIR / "candidate_monthly_coverage_company_flags.csv"
COVERAGE_REPORT_FILE = BASE_DIR / "candidate_monthly_coverage_report.json"

OUTPUT_LATEST_COMPANY = OUT_DIR / "financial_features_candidate_latest_company.csv"
OUTPUT_T1_MODEL_READY = OUT_DIR / "financial_features_candidate_latest_model_ready_t1.csv"
OUTPUT_PROXY_UNLABELLED = OUT_DIR / "financial_features_candidate_latest_proxy_unlabelled.csv"
OUTPUT_SUMMARY = OUT_DIR / "financial_features_candidate_latest_company_summary.csv"
OUTPUT_MISSINGNESS = OUT_DIR / "financial_features_candidate_latest_missingness.csv"
OUTPUT_GROUP_SUMMARY = OUT_DIR / "financial_features_candidate_latest_group_summary.csv"
OUTPUT_FEATURE_COLUMNS = OUT_DIR / "model_feature_columns.json"

require_exists(FEATURES_FILE, "candidate financial features file")
require_exists(COVERAGE_FLAGS_FILE, "coverage flags file")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOCAL_DATA_ROOT:", LOCAL_DATA_ROOT)
print("Input features:", FEATURES_FILE)
print("Coverage flags:", COVERAGE_FLAGS_FILE)
print("Output dir:", OUT_DIR)


## 1. 读取数据与基础检查


In [ ]:
features = pd.read_csv(FEATURES_FILE, dtype={"CompanyNumber": str})
coverage_flags = pd.read_csv(COVERAGE_FLAGS_FILE, dtype={"CompanyNumber": str})

if COVERAGE_REPORT_FILE.exists():
    coverage_report = json.loads(COVERAGE_REPORT_FILE.read_text(encoding="utf-8"))
else:
    coverage_report = {}

features["CompanyNumber"] = features["CompanyNumber"].astype(str).str.upper().str.zfill(8)
coverage_flags["CompanyNumber"] = coverage_flags["CompanyNumber"].astype(str).str.upper().str.zfill(8)

required_cols = [
    "CompanyNumber",
    "CompanyName",
    "period_end",
    "source_zip",
    "internal_filename",
    "financial_core_score",
    "financial_conservative_score",
    "financial_evidence_tier",
    "turnover",
    "observed_segment_from_turnover",
]
missing_required = [c for c in required_cols if c not in features.columns]
if missing_required:
    raise ValueError(f"Missing required columns in features file: {missing_required}")

print("Feature rows:", len(features))
print("Unique companies in feature rows:", features["CompanyNumber"].nunique())
print("Coverage flag rows:", len(coverage_flags))
display(features.head())
display(coverage_flags.head())


## 2. 选择每家公司 latest filing


In [ ]:
def normalise_period(value):
    value = "" if pd.isna(value) else str(value)
    digits = "".join(ch for ch in value if ch.isdigit())
    if len(digits) >= 8:
        return int(digits[:8])
    return -1


features["period_end_numeric"] = features["period_end"].map(normalise_period)
features["period_end_date"] = pd.to_datetime(
    features["period_end_numeric"].where(features["period_end_numeric"].ge(0)).astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce",
)

sort_cols = ["CompanyNumber", "period_end_numeric", "source_zip", "internal_filename"]
features_sorted = features.sort_values(sort_cols).reset_index(drop=True)
latest = features_sorted.drop_duplicates(subset=["CompanyNumber"], keep="last").copy()

filing_agg = (
    features_sorted.groupby("CompanyNumber", dropna=False)
    .agg(
        company_account_file_rows=("internal_filename", "count"),
        company_first_period=("period_end_numeric", "min"),
        company_latest_period=("period_end_numeric", "max"),
        company_observed_turnover_file_count=("turnover", lambda s: int(s.notna().sum())),
        company_has_any_observed_turnover=("turnover", lambda s: bool(s.notna().any())),
        company_max_financial_core_score=("financial_core_score", "max"),
        company_max_financial_conservative_score=("financial_conservative_score", "max"),
    )
    .reset_index()
)

turnover_rows = features_sorted[features_sorted["turnover"].notna()].copy()
if len(turnover_rows):
    latest_turnover = (
        turnover_rows.sort_values(sort_cols)
        .drop_duplicates(subset=["CompanyNumber"], keep="last")
        [[
            "CompanyNumber",
            "turnover",
            "period_end_numeric",
            "observed_segment_from_turnover",
            "source_zip",
            "internal_filename",
        ]]
        .rename(columns={
            "turnover": "company_latest_observed_turnover_any_filing",
            "period_end_numeric": "company_latest_turnover_period",
            "observed_segment_from_turnover": "company_latest_observed_segment_any_filing",
            "source_zip": "company_latest_turnover_source_zip",
            "internal_filename": "company_latest_turnover_internal_filename",
        })
    )
else:
    latest_turnover = pd.DataFrame(columns=[
        "CompanyNumber",
        "company_latest_observed_turnover_any_filing",
        "company_latest_turnover_period",
        "company_latest_observed_segment_any_filing",
        "company_latest_turnover_source_zip",
        "company_latest_turnover_internal_filename",
    ])

latest = latest.merge(filing_agg, on="CompanyNumber", how="left")
latest = latest.merge(latest_turnover, on="CompanyNumber", how="left")

coverage_keep = [
    c for c in [
        "CompanyNumber",
        "has_bulk_account",
        "matched_file_count",
        "matched_zip_count",
        "first_period",
        "latest_period",
    ]
    if c in coverage_flags.columns
]
latest = latest.merge(
    coverage_flags[coverage_keep].drop_duplicates(subset=["CompanyNumber"]),
    on="CompanyNumber",
    how="left",
    suffixes=("", "_coverage"),
)

latest["latest_has_observed_turnover"] = latest["turnover"].notna()
latest["latest_observed_turnover"] = latest["turnover"]
latest["latest_observed_segment_from_turnover"] = latest["observed_segment_from_turnover"].fillna("")
latest["model_label_available_latest"] = latest["latest_has_observed_turnover"] & latest[
    "latest_observed_segment_from_turnover"
].isin(["BB", "SME", "MidCorporate"])
latest["model_label_bb_vs_non_bb"] = np.where(
    latest["model_label_available_latest"],
    np.where(latest["latest_observed_segment_from_turnover"].eq("BB"), "BB", "non_BB"),
    "",
)


# Derived financial structure features.
# These features describe how the financial scale is composed; they do not use turnover.
def safe_ratio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    denominator = denominator.where(denominator.abs() > 1e-9)
    return numerator / denominator


def signed_log1p_series(series):
    values = pd.to_numeric(series, errors="coerce")
    return np.sign(values) * np.log1p(values.abs())


def log_per_employee(value_col):
    employees = pd.to_numeric(latest["employees"], errors="coerce")
    employees = employees.where(employees > 0)
    return signed_log1p_series(latest[value_col]) - np.log1p(employees)


latest["cash_to_current_assets"] = safe_ratio(latest["cash"], latest["current_assets"])
latest["debtors_to_current_assets"] = safe_ratio(latest["debtors"], latest["current_assets"])
latest["creditors_to_current_assets"] = safe_ratio(latest["creditors_total"], latest["current_assets"])
latest["net_assets_to_current_assets"] = safe_ratio(latest["net_assets_liabilities"], latest["current_assets"])
latest["equity_to_current_assets"] = safe_ratio(latest["equity"], latest["current_assets"])
latest["borrowings_to_current_assets"] = safe_ratio(latest["borrowings"], latest["current_assets"])
latest["profit_loss_to_current_assets"] = safe_ratio(latest["profit_loss"], latest["current_assets"])

latest["log_current_assets_per_employee"] = log_per_employee("current_assets")
latest["log_creditors_per_employee"] = log_per_employee("creditors_total")
latest["log_debtors_per_employee"] = log_per_employee("debtors")
latest["log_cash_per_employee"] = log_per_employee("cash")
latest["log_profit_loss_per_employee"] = log_per_employee("profit_loss")

current_assets_q75 = pd.to_numeric(latest["current_assets"], errors="coerce").quantile(0.75)
log_assets_per_employee_q75 = latest["log_current_assets_per_employee"].quantile(0.75)

latest["cash_heavy_flag"] = latest["cash_to_current_assets"].ge(0.50)
latest["debtor_heavy_flag"] = latest["debtors_to_current_assets"].ge(0.50)
latest["creditor_heavy_flag"] = latest["creditors_to_current_assets"].ge(0.50)
latest["asset_heavy_flag"] = latest["log_current_assets_per_employee"].ge(log_assets_per_employee_q75)
latest["low_employee_high_asset_flag"] = (
    pd.to_numeric(latest["employees"], errors="coerce").fillna(0).le(2)
    & pd.to_numeric(latest["current_assets"], errors="coerce").ge(current_assets_q75)
)
latest["negative_equity_flag"] = pd.to_numeric(latest["equity"], errors="coerce").lt(0)
latest["negative_net_assets_flag"] = pd.to_numeric(latest["net_assets_liabilities"], errors="coerce").lt(0)

derived_ratio_features = [
    "cash_to_current_assets",
    "debtors_to_current_assets",
    "creditors_to_current_assets",
    "net_assets_to_current_assets",
    "equity_to_current_assets",
    "borrowings_to_current_assets",
    "profit_loss_to_current_assets",
    "log_current_assets_per_employee",
    "log_creditors_per_employee",
    "log_debtors_per_employee",
    "log_cash_per_employee",
    "log_profit_loss_per_employee",
]

derived_binary_features = [
    "cash_heavy_flag",
    "debtor_heavy_flag",
    "creditor_heavy_flag",
    "asset_heavy_flag",
    "low_employee_high_asset_flag",
    "negative_equity_flag",
    "negative_net_assets_flag",
]

print("Latest company rows:", len(latest))
print("Unique companies:", latest["CompanyNumber"].nunique())
print("Duplicated companies:", int(latest["CompanyNumber"].duplicated().sum()))
display(latest[[
    "CompanyNumber",
    "CompanyName",
    "period_end",
    "latest_has_observed_turnover",
    "latest_observed_turnover",
    "latest_observed_segment_from_turnover",
    "company_has_any_observed_turnover",
    "company_observed_turnover_file_count",
    "financial_evidence_tier",
    "financial_core_score",
    "financial_conservative_score",
    "cash_to_current_assets",
    "creditors_to_current_assets",
    "log_current_assets_per_employee",
    "asset_heavy_flag",
    "low_employee_high_asset_flag",
]].head(20))


## 3. 建模字段与子集定义


In [ ]:
ID_COLUMNS = [
    "CompanyNumber",
    "CompanyName",
    "primary_sector",
    "Accounts_AccountCategory",
    "account_category_group",
    "period_end",
    "period_end_iso",
    "source_zip",
    "internal_filename",
]

LABEL_COLUMNS = [
    "latest_has_observed_turnover",
    "latest_observed_turnover",
    "latest_observed_segment_from_turnover",
    "model_label_available_latest",
    "model_label_bb_vs_non_bb",
    "company_has_any_observed_turnover",
    "company_latest_observed_turnover_any_filing",
    "company_latest_observed_segment_any_filing",
]

NUMERIC_FEATURE_COLUMNS = [
    "financial_core_score",
    "financial_conservative_score",
    "financial_core_score_raw",
    "financial_conservative_score_raw",
    "available_core_balance_field_count",
    "available_proxy_field_count",
    "financial_data_quality_score",
    "current_assets",
    "net_assets_liabilities",
    "equity",
    "cash",
    "debtors",
    "creditors_total",
    "employees",
    "profit_loss",
    "borrowings",
    "cash_to_current_assets",
    "debtors_to_current_assets",
    "creditors_to_current_assets",
    "net_assets_to_current_assets",
    "equity_to_current_assets",
    "borrowings_to_current_assets",
    "profit_loss_to_current_assets",
    "log_current_assets_per_employee",
    "log_creditors_per_employee",
    "log_debtors_per_employee",
    "log_cash_per_employee",
    "log_profit_loss_per_employee",
    "company_account_file_rows",
    "matched_file_count",
    "matched_zip_count",
]

BINARY_FEATURE_COLUMNS = [
    "has_current_assets",
    "has_net_assets_liabilities",
    "has_equity",
    "has_cash",
    "has_debtors",
    "has_creditors_total",
    "has_employees",
    "has_balance_sheet_core",
    "has_employee_count",
    "cash_heavy_flag",
    "debtor_heavy_flag",
    "creditor_heavy_flag",
    "asset_heavy_flag",
    "low_employee_high_asset_flag",
    "negative_equity_flag",
    "negative_net_assets_flag",
]

CATEGORICAL_FEATURE_COLUMNS = [
    "Accounts_AccountCategory",
    "account_category_group",
    "primary_sector",
    "financial_evidence_tier",
]

MODEL_FEATURE_COLUMNS = (
    NUMERIC_FEATURE_COLUMNS
    + BINARY_FEATURE_COLUMNS
    + CATEGORICAL_FEATURE_COLUMNS
)

existing_model_features = [c for c in MODEL_FEATURE_COLUMNS if c in latest.columns]
missing_model_features = [c for c in MODEL_FEATURE_COLUMNS if c not in latest.columns]

t1_model_ready = latest[
    latest["model_label_available_latest"].fillna(False)
    & latest["latest_observed_segment_from_turnover"].isin(["BB", "SME", "MidCorporate"])
].copy()

proxy_unlabelled = latest[
    ~latest["model_label_available_latest"].fillna(False)
    & latest["financial_evidence_tier"].isin([
        "T2_balance_sheet_rich",
        "T3_balance_sheet_partial",
        "T4_account_category_only",
    ])
].copy()

print("Existing model features:", len(existing_model_features))
print("Missing model features:", missing_model_features)
print("T1 model-ready latest companies:", len(t1_model_ready))
print("Proxy unlabelled latest companies:", len(proxy_unlabelled))
display(t1_model_ready["latest_observed_segment_from_turnover"].value_counts(dropna=False).to_frame("companies"))
display(proxy_unlabelled["financial_evidence_tier"].value_counts(dropna=False).to_frame("companies"))


## 4. Summary tables


In [ ]:
def rate(numerator, denominator):
    return float(numerator / denominator) if denominator else 0.0


summary_rows = [
    {"metric": "candidate_latest_company_rows", "value": len(latest)},
    {"metric": "unique_companies", "value": latest["CompanyNumber"].nunique()},
    {"metric": "latest_t1_model_ready_companies", "value": len(t1_model_ready)},
    {"metric": "latest_t1_model_ready_rate", "value": rate(len(t1_model_ready), len(latest))},
    {"metric": "latest_bb_companies", "value": int(t1_model_ready["latest_observed_segment_from_turnover"].eq("BB").sum())},
    {"metric": "latest_non_bb_companies", "value": int(t1_model_ready["latest_observed_segment_from_turnover"].isin(["SME", "MidCorporate"]).sum())},
    {"metric": "company_any_turnover_companies", "value": int(latest["company_has_any_observed_turnover"].fillna(False).sum())},
    {"metric": "company_any_turnover_rate", "value": rate(int(latest["company_has_any_observed_turnover"].fillna(False).sum()), len(latest))},
    {"metric": "proxy_unlabelled_companies", "value": len(proxy_unlabelled)},
    {"metric": "parse_ready_rows_without_duplicate_company", "value": int(latest["CompanyNumber"].duplicated().sum() == 0)},
]
summary_df = pd.DataFrame(summary_rows)

segment_summary = (
    t1_model_ready.groupby("latest_observed_segment_from_turnover", dropna=False)
    .agg(
        companies=("CompanyNumber", "count"),
        turnover_median=("latest_observed_turnover", "median"),
        core_score_median=("financial_core_score", "median"),
        conservative_score_median=("financial_conservative_score", "median"),
        core_field_count_median=("available_core_balance_field_count", "median"),
    )
    .reset_index()
)

group_summary_parts = []
for group_col in ["Accounts_AccountCategory", "primary_sector", "financial_evidence_tier"]:
    if group_col not in latest.columns:
        continue
    part = (
        latest.groupby(group_col, dropna=False)
        .agg(
            companies=("CompanyNumber", "count"),
            latest_t1_companies=("model_label_available_latest", "sum"),
            any_turnover_companies=("company_has_any_observed_turnover", "sum"),
            core_score_median=("financial_core_score", "median"),
            conservative_score_median=("financial_conservative_score", "median"),
            core_field_count_median=("available_core_balance_field_count", "median"),
        )
        .reset_index()
        .rename(columns={group_col: "group"})
    )
    part["summary_level"] = group_col
    part["latest_t1_rate"] = part["latest_t1_companies"] / part["companies"]
    part["any_turnover_rate"] = part["any_turnover_companies"] / part["companies"]
    group_summary_parts.append(part)

group_summary = pd.concat(group_summary_parts, ignore_index=True, sort=False)

missingness = []
for col in existing_model_features + LABEL_COLUMNS:
    if col not in latest.columns:
        continue
    missingness.append({
        "column": col,
        "missing_count": int(latest[col].isna().sum()),
        "missing_rate": float(latest[col].isna().mean()),
        "non_missing_count": int(latest[col].notna().sum()),
    })
missingness_df = pd.DataFrame(missingness).sort_values(["missing_rate", "column"], ascending=[False, True])

display(summary_df)
display(segment_summary)
display(group_summary.head(30))
display(missingness_df.head(30))


## 5. 写出 company-level latest 文件


In [ ]:
latest.to_csv(OUTPUT_LATEST_COMPANY, index=False, encoding="utf-8-sig")
t1_model_ready.to_csv(OUTPUT_T1_MODEL_READY, index=False, encoding="utf-8-sig")
proxy_unlabelled.to_csv(OUTPUT_PROXY_UNLABELLED, index=False, encoding="utf-8-sig")
summary_df.to_csv(OUTPUT_SUMMARY, index=False, encoding="utf-8-sig")
missingness_df.to_csv(OUTPUT_MISSINGNESS, index=False, encoding="utf-8-sig")
group_summary.to_csv(OUTPUT_GROUP_SUMMARY, index=False, encoding="utf-8-sig")

feature_config = {
    "id_columns": ID_COLUMNS,
    "label_columns": LABEL_COLUMNS,
    "numeric_feature_columns": NUMERIC_FEATURE_COLUMNS,
    "binary_feature_columns": BINARY_FEATURE_COLUMNS,
    "categorical_feature_columns": CATEGORICAL_FEATURE_COLUMNS,
    "model_feature_columns": MODEL_FEATURE_COLUMNS,
    "existing_model_features": existing_model_features,
    "missing_model_features": missing_model_features,
    "label_definition": {
        "primary_training_label": "latest_observed_segment_from_turnover",
        "bb_binary_label": "model_label_bb_vs_non_bb",
        "label_available_flag": "model_label_available_latest",
        "note": "Primary model labels are derived from the latest filing only.",
    },
    "input_files": {
        "features": str(FEATURES_FILE),
        "coverage_flags": str(COVERAGE_FLAGS_FILE),
        "coverage_report": str(COVERAGE_REPORT_FILE),
    },
    "output_files": {
        "latest_company": str(OUTPUT_LATEST_COMPANY),
        "t1_model_ready": str(OUTPUT_T1_MODEL_READY),
        "proxy_unlabelled": str(OUTPUT_PROXY_UNLABELLED),
        "summary": str(OUTPUT_SUMMARY),
        "missingness": str(OUTPUT_MISSINGNESS),
        "group_summary": str(OUTPUT_GROUP_SUMMARY),
    },
}

OUTPUT_FEATURE_COLUMNS.write_text(json.dumps(feature_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("Wrote:")
for path in [
    OUTPUT_LATEST_COMPANY,
    OUTPUT_T1_MODEL_READY,
    OUTPUT_PROXY_UNLABELLED,
    OUTPUT_SUMMARY,
    OUTPUT_MISSINGNESS,
    OUTPUT_GROUP_SUMMARY,
    OUTPUT_FEATURE_COLUMNS,
]:
    print(path)


## 6. 给第二个建模 notebook 的输入

第二个 notebook 建议直接读取：

```text
financial_features_candidate_latest_company.csv
model_feature_columns.json
```

建模训练集优先使用：

```text
financial_features_candidate_latest_model_ready_t1.csv
```

应用 / scoring 数据可使用：

```text
financial_features_candidate_latest_proxy_unlabelled.csv
```

第一版 baseline 建议：

```text
Task A: BB vs non_BB
Task B: BB / SME / MidCorporate multiclass sanity check
```
